In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
model = AutoModelForCausalLM.from_pretrained("sshleifer/tiny-gpt2")

In [3]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

# Ensure pad token exists
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Optional: reproducible sampling
# torch.manual_seed(0)

def generate(prompt, max_new_tokens=40, temperature=0.8, top_p=0.95, do_sample=True):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    gen_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Return only newly generated tokens
    new_tokens = gen_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Q&A style:")
print("Prompt: Question: What is 2+2? Answer:")
print("Completion:", generate("Question: What is 2+2? Answer:"))

print("\nFreeform style:")
print("Completion:", generate("Once upon a time"))

Q&A style:
Prompt: Question: What is 2+2? Answer:
Completion: oother vendors conservationting Jrpressootherimura confir Motorolareement Participation reviewing Participation Prob004004reementreement dispatch ESV scalp Habit circumcisediken confir Prob MotorolaJD TA ESV scalp Habit Jr stairs autonomymediately HancockRocket ONE

Freeform style:
Completion:  rubbing653 praying deflect Dreams Boone rubbing653 factors brutalityobl representationspublic workshopsMost praying Redux Television Medic perhaps workshopsOutside Wheels Wheels clearer factors courtyard equate incarcer SingaporeGyobl brutality deflect boils Boone membership membershipGy workshops


In [4]:
# "hidden_size" is the embedding (model) dimension for GPT-2 family
hs_cfg = getattr(model.config, "hidden_size", None)
n_embd = getattr(model.config, "n_embd", None)  # GPT-2-specific name
print("config.hidden_size:", hs_cfg)
print("config.n_embd:", n_embd)
print("embedding/hidden size used:", hs_cfg or n_embd)

config.hidden_size: 2
config.n_embd: 2
embedding/hidden size used: 2


In [5]:
emb = model.get_input_embeddings().weight  # shape: (vocab_size, hidden_size)
vocab_size, hidden_size = emb.shape
print("vocab_size:", vocab_size)
print("embedding (hidden) size:", hidden_size)

vocab_size: 50257
embedding (hidden) size: 2


In [8]:
import torch
inputs = tokenizer("hi", return_tensors="pt").to(device)
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True, use_cache=False)
h = out.hidden_states[-1]  # shape: (batch, seq_len, hidden_size)
print("last hidden state shape:", tuple(h.shape))
print("embedding/hidden size:", h.shape[-1])

last hidden state shape: (1, 1, 2)
embedding/hidden size: 2
